In [30]:
param = {
  "test": "power", # ['type1error', 'power']
  "sample_size": 200, # [200, 400, 600, 800, 1000]
  "batch_size": 128, # [32, 64, 128, 256]
  "z_dim": 1, # [5, 50, 250]
  "dx": 1,
  "dy": 1,
  "n_test": 100, # [200, 2000] number of trails to run
  "epochs_num": 100, # [1000, 1500] number of epoch
  "eps_std" : 1.0, # epsilon std
  "dist_z" : 'gaussian', # ['laplace', 'gaussian']
  "alpha_x": 0.20, # only used under alternative [0.05, 0.10, 0.15, 0.20, 0.25]
  "m_value": 100, # [100, 200]
  "k_value": 8, # [2, 4, 6, 8] fold number
  "j_value": 1000, # [1000, 2000]
  "noise_dimension": 5, # [5, 10, 20]
  "noise_dimension_type": "normal",  # ["normal", "unif", "Cauchy"]
  "noise_dimension_var": 1, # [1, 4, 9, 16] the variance of the input noise if it is normal
  "hidden_layer_size": 1024, # [64, 128, 256, 512, 1024]
  "normal_ini": False, # [True, False]
  "preprocess": 'scale', # ['normalize', 'scale', 'None']
  "G_lr": 7e-5, # [5e-6, 1e-5, 2e-5] generator lr 1e-3
  "alpha": 0.1, # significance level 1
  "alpha1": 0.05, # significance level 2
  "set_seeds": 0,
  "using_orcale": False,
  "lambda_1": 1, # loss with Laplace kernel
  "lambda_2": 1,  # loss with Gaussian kernel
  "using_Gen": '1',  # ['1', '2'], types of generator "1" is fully connect, "2" is non fully
  "boor_rv_type":  'gaussian', # ['rademacher', 'gaussian']
  "wgt_decay": 1e-5, # weight decay for adam optimizer L2 regularization parameter
  "lambda_3": 1e-4, # L1 regularization parameter for rest of the layers
  "lambda_4": 2e-5, # L1 regularization parameter for the first layer
  "drop_out_p": 0.2, #  probability of an element to be zeroed. Default: 0.5, best 0.2
  "is_sparse": True, # [True, False], using sparse Ax, Ay
  "sparse_ratio": 0.05, # the ratio of non-zero elements in Ax, Ay
  "lambda_mean": 0.3, # the tuning parameter for first moment alignment
  "mean_samples": 20 # the number of samples used for first moment alignment
}


import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
from zmq import device
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools
from tqdm import tqdm


# Move model on GPU if available
enable_cuda = True
device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')


def generate_samples_random(size=1000, sType='H0', dx=1, dy=1, dz=1, nstd=1.0, alpha_x=0.05,
               preprocess="None", dist_z='gaussian'):
    '''
    Generate CI,I or NI post-nonlinear samples
    1. Z is independent Gaussian or Laplace
    2. X = f1(<a,Z> + b + noise) and Y = f2(<c,Z> + d + noise) in case of CI
    Arguments:
        size : number of samples
        sType: CI, I, or NI
        dx: Dimension of X
        dy: Dimension of Y
        dz: Dimension of Z
        nstd: noise standard deviation
        we set f1 to be sin function and f2 to be cos function.
    Output:
        Samples X, Y, Z
    '''

    num = size

    error_generator_x = TD.MultivariateNormal(
        torch.zeros(dx), 1 * torch.eye(dx))

    error_generator_y = TD.MultivariateNormal(
        torch.zeros(dy), 1 * torch.eye(dy))

    if dist_z == 'gaussian':
        z_generator = TD.MultivariateNormal( torch.zeros(dz), 1 * torch.eye(dz))
        Z = z_generator.sample((num,))

    elif dist_z == 'laplace':
        z_generator = TD.MultivariateNormal( torch.zeros(dz), 1 * torch.eye(dz))
        Z = z_generator.sample((num,))

    m = TD.Bernoulli(torch.tensor([alpha_x]))

    Axy = torch.ones((dx, dy)) * alpha_x

    if sType == 'H0':
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        X = Z + nstd * error_x
        Y = Z + nstd * error_y
    else:
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        delta = m.sample((num,))
        X = Z + (1 - delta) * nstd * error_x + delta * nstd * error_y
        Y = Z + nstd * error_y


    return X, Y, Z

def generate_samples_from_fixed_Z_random(Z, size=1000, sType='H0', dx=1, dy=1, dz=1, nstd=1.0, alpha_x=0.05,
                     normalize=True, seed=None, dist_z='gaussian'):
    '''
    Generate CI,I or NI post-nonlinear samples for fixed Z
    1. Z is independent Gaussian or Laplace
    2. X = f1(<a,Z> + b + noise) and Y = f2(<c,Z> + d + noise) in case of CI
    Arguments:
        size : number of samples
        sType: CI, I, or NI
        dx: Dimension of X
        dy: Dimension of Y
        dz: Dimension of Z
        nstd: noise standard deviation
        we set f1 to be sin function and f2 to be cos function.
    Output:
        Samples X, Y, Z
    '''
    num = size

    error_generator_x = TD.MultivariateNormal(
        torch.zeros(dx), 1 * torch.eye(dx))

    error_generator_y = TD.MultivariateNormal(
        torch.zeros(dy), 1 * torch.eye(dy))

    Axy = torch.ones((dx, dy)) * alpha_x
    m = TD.Bernoulli(torch.tensor([alpha_x]))
    
    if sType == 'H0':
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        X = Z + nstd * error_x
        Y = Z + nstd * error_y
    else:
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        delta = m.sample((num,))
        X = Z + (1 - delta) * nstd * error_x + delta * nstd * error_y
        Y = Z + nstd * error_y

    return X, Y

def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch, x_torch, y_torch, z_torch, sigma_w, sigma_u = 1, sigma_v = 1,
                       boor_rv_type = "gaussian"):
    """
    Compute the p-value

    Input:
    - boot_num: Integer giving the number of bootstrap samples.
    - M: Integer giving the number of training samples per batch.
    - n: Integer giving the number of training samples.
    - gen_x_all_torch: PyTorch Tensor (batch_size, M) of generated data of X.
    - gen_y_all_torch: PyTorch Tensor (batch_size, M) of generated data of Y.
    - x_torch: PyTorch Tensor (batch_size) of training input X.
    - y_torch: PyTorch Tensor (batch_size) of training input Y.
    - z_torch: PyTorch Tensor (batch_size, dimension_Z) of training input Z
    - sigma_w: Float of the bandwith of the Laplace kernel.
    - sigma_u: Float of the bandwith of the Laplace kernel.
    - sigma_v: Float of the bandwith of the Laplace kernel.
    - boor_rv_type: "rademacher" or "gaussian", specifying the reference distribution.

    Output:
    - p_value: Float giving the p-value.
    """
    w_mx = torch.linalg.vector_norm(z_torch.repeat(n,1,1) - torch.swapaxes(z_torch.repeat(n,1,1), 0, 1), ord = 1, dim = 2)
    w_mx = torch.exp(-w_mx/sigma_w)

    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1,n) - y_torch.repeat(1,n).T)/sigma_u)
    u_mx_2 = torch.mean(torch.exp(-torch.abs(gen_y_all_torch.repeat(n,1,1) - y_torch.repeat(1, n).reshape(n,n,1))/sigma_u), dim = 2)
    u_mx_3 = u_mx_2.T

    gen_y_all_torch_rep = gen_y_all_torch.repeat(n,1,1)

    temp_mx = gen_y_all_torch_rep[:,:,0].T
    sum_mx = torch.mean(torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n,n,1))/sigma_u), dim = 2)

    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1,n) - x_torch.repeat(1,n).T)/sigma_v)
    v_mx_2 = torch.mean(torch.exp(-torch.abs(gen_x_all_torch.repeat(n,1,1) - x_torch.repeat(1, n).reshape(n,n,1))/sigma_v), dim = 2)
    v_mx_3 = v_mx_2.T

    gen_x_all_torch_rep = gen_x_all_torch.repeat(n,1,1)

    temp2_mx = gen_x_all_torch_rep[:,:,0].T
    sum2_mx = torch.mean(torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n,n,1))/sigma_v), dim = 2)

    for i in range(1, M):
      temp_mx = gen_y_all_torch_rep[:,:,i].T
      temp_add_mx = torch.mean(torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n,n,1))/sigma_u), dim = 2)
      sum_mx = sum_mx + temp_add_mx

      temp2_mx = gen_x_all_torch_rep[:,:,i].T
      temp2_add_mx = torch.mean(torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n,n,1))/sigma_v), dim = 2)
      sum2_mx = sum2_mx + temp2_add_mx

    u_mx_4 = 1/M*sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4
    v_mx_4 = 1/M*sum2_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + v_mx_4

    FF_mx = u_mx * v_mx *w_mx * (1-torch.eye(n).to(device))

    stat = 1/(n-1) * torch.sum(FF_mx).item()

    boottemp = np.array([])
    if boor_rv_type == "rademacher":
      eboot = torch.sign(torch.randn(n, boot_num)).to(device)
    elif boor_rv_type == "gaussian":
      eboot = torch.randn(n, boot_num).to(device)
    for bb in range(boot_num):
      random_mx = torch.matmul(eboot[:,bb].reshape(-1,1), eboot[:,bb].reshape(-1,1).T)
      bootmatrix = FF_mx * random_mx
      stat_boot = 1/(n-1) * torch.sum(bootmatrix).item()
      boottemp = np.append(boottemp, stat_boot)
    return stat, boottemp

class DatasetSelect(Dataset):
    """
    Create a DatasetSelect object to generate the DataLoader in the learning process.

    Input:
    - X: PyTorch Tensor of shape (N, input_dimension) giving the training data of X.
    - Y: PyTorch Tensor of shape (N, output_dimension) giving the training data of Y.
    - Z: PyTorch Tensor of shape (N, output_dimension) giving the training data of Z.
    """


    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]

# Create a DataLoader for given (X, Y)

class DatasetSelect_GAN(torch.utils.data.Dataset):
  """
    Create a DatasetSelect object to generate the DataLoader in the learning process.

    Input:
    - X: PyTorch Tensor of shape (N, input_dimension) giving the training data of X.
    - Y: PyTorch Tensor of shape (N, output_dimension) giving the training data of Y.
    - batch_size: Integer giving the batch size.
  """

  def __init__(self, X, Y, Z, batch_size):
    self.X_real = X
    self.Y_real = Y
    self.Z_real = Z
    self.batch_size = batch_size
    self.sample_size = X.shape[0]

  def __len__(self):
    return self.sample_size

  def __getitem__(self, index):
    return self.X_real[index], self.Y_real[index], self.Z_real[index], self.Z_real[(self.batch_size+index) % self.sample_size]

# Create a DataLoader for given (X, Y)

class DatasetSelect_GAN_ver2(torch.utils.data.Dataset):
  """
    Create a DatasetSelect object to generate the DataLoader in the learning process.

    Input:
    - X: PyTorch Tensor of shape (N, input_dimension) giving the training data of X.
    - Y: PyTorch Tensor of shape (N, output_dimension) giving the training data of Y.
    - batch_size: Integer giving the batch size.
  """

  def __init__(self, Y, Z, batch_size):
    self.Y_real = Y
    self.Z_real = Z
    self.batch_size = batch_size
    self.sample_size = Z.shape[0]

  def __len__(self):
    return self.sample_size

  def __getitem__(self, index):
    return self.Y_real[index], self.Z_real[index]

##### Auxilliary functions #####

def sample_noise(sample_size, noise_dimension, noise_type, input_var = 1):
    """
    Generate a PyTorch Tensor of random noise from the specified reference distribution.

    Input:
    - sample_size: the sample size of noise to generate.
    - noise_dimension: the dimension of noise to generate.
    - noise_type: "normal", "unif" or "Cauchy", giving the reference distribution.

    Output:
    - A PyTorch Tensor of shape (sample_size, noise_dimension).
    """

    if (noise_type == "normal"):
      noise_generator = TD.MultivariateNormal(
        torch.zeros(noise_dimension).to(device), input_var * torch.eye(noise_dimension).to(device))

      Z = noise_generator.sample((sample_size,))
    if (noise_type == "unif"):
      Z = torch.rand(sample_size, noise_dimension)
    if (noise_type == "Cauchy"):
      Z = TD.Cauchy(torch.tensor([0.0]), torch.tensor([1.0])).sample((sample_size, noise_dimension)).squeeze(2)

    return Z

##### GAN architecture #####

class Generator(torch.nn.Module):
    """
    Specify the neural network architecture of the Generator.

    Here, we consider a FNN with a fully connected hidden layer with a width of 50,
    which is followed by a Leaky ReLU activation. The coefficient of Leaky ReLU needs to be
    specified. Batch normalization may be added prior to the activation function.
    The output layer a fully connected layer without activation.

    Inputs:
    - input_dimension: Integer giving the dimension of input Z.
    - output_dimension: Integer giving the dimension of output X or Y.
    - noise_dimension: Integer giving the dimension of random noise.
    - hidden_layer_size: Integer giving the size of the hidden layer of the generator.
    - BN_type: 'True' or 'False' specifying whether batch normalization is included.
    - ReLU_coef: Scalar giving the coefficient of the Leaky ReLU layer.
    - drop_out_p: Float giving the dropout probability.
    - drop_input: Boolean specifying whether to add dropout to the input layer.

    Returns:
    - x: PyTorch Tensor containing the (output_dimension,) output of the generator.
    """

    def __init__(self, input_dimension, output_dimension, noise_dimension, hidden_layer_size, BN_type, ReLU_coef, drop_out_p,
                 drop_input = False):
      super(Generator, self).__init__()
      self.BN_type = BN_type
      self.ReLU_coef = ReLU_coef
      self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
      if BN_type:
        self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
        self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
        self.BN3 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
      self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)
      self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
      self.fc3 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
      self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)
      self.sigmoid = torch.nn.Sigmoid()
      self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
      self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
      self.drop_out2 = torch.nn.Dropout(p=drop_out_p)
      self.drop_out3 = torch.nn.Dropout(p=drop_out_p)
      self.drop_input = drop_input

    def forward(self, x):
      if self.BN_type:
        if self.drop_input:
            x = self.drop_out0(x)
        x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
        x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
        # x = self.drop_out3(self.leakyReLU1(self.BN3(self.fc3(x))))
        x = self.fc_last(x)
      else:
        if self.drop_input:
            x = self.drop_out0(x)
        x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
        x = self.drop_out2(self.leakyReLU1(self.fc2(x)))
        # x = self.drop_out3(self.leakyReLU1(self.fc3(x)))
        x = self.fc_last(x)
      return x

class NonFullyConnected_1(torch.nn.Module):

  def __init__(self, size_in, size_out, m, bias = True):
    super(NonFullyConnected_1, self).__init__()
    self.linear = torch.nn.Linear(m*size_in, m*size_out, bias = bias).to(device)
    self.mask = functools.reduce(torch.block_diag,[torch.ones(size_out, size_in) for i in range(m)]).to(device)

  def forward(self, x):

    self.linear.weight.data *= self.mask
    return self.linear(x)

class Generator_2(torch.nn.Module):
    def __init__(
            self,
            input_dimension,
            output_dimension,
            noise_dimension,
            hidden_layer_size,
            BN_type,
            ReLU_coef,
            hidden_layer_depth = 1,
            ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)
        self.linear_layers_from_input = torch.nn.Linear(self.input_dimension, ntargets_k*self.hidden_layer_sizes[0])

        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0],self.hidden_layer_sizes[0], ntargets_k)
            for i in range(len(self.hidden_layer_sizes))
        ])
        # self.linear8 = torch.nn.Linear(self.hidden_layer_sizes[0]*ntargets_k, self.hidden_layer_sizes[0]*ntargets_k)
        # self.linear8.weight = torch.nn.Parameter(torch.eye(self.hidden_layer_sizes[0]*ntargets_k), requires_grad=False)
        self.linear8 = torch.nn.Linear(self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
    def forward(self, input):
        if self.BN_type:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(self.BN1(output))
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(self.BN1(output))
        else:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(output)
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(output)

        return self.linear8(output) # torch.mean(self.linear8(output)).reshape(1)

##### Training procedures #####


def find_loss(Y, hat_Y, Z, sigma_z, sigma_y):
    """
    Compute the MMD loss via Laplace kernel.

    Inputs:
    - y_torch: PyTorch Tensor (batch_size) of training input. (X or Y)
    - gen_y_all_torch: PyTorch Tensor (batch_size, M) of generated data.
    - z_torch: PyTorch Tensor (batch_size, dimension_Z) of training input Z.
    - sigma_w: Float of the bandwith of the kernel.
    - sigma_u: Float of the bandwith of the kernel.
    - M: Number of training samples per batch.

    Outputs:
    - loss: PyTorch Tensor containing the MMD loss.
    """

    n = Z.shape[0]
    mx_1_1 = torch.exp(-torch.abs(Y.repeat(1,n) - Y.repeat(1,n).T)/sigma_y)
    mx_1_2 = torch.linalg.vector_norm(Z.repeat(n,1,1) - torch.swapaxes(Z.repeat(n,1,1), 0, 1), ord = 1, dim = 2)
    # sigma = torch.median(mx_1_2)
    mx_1_2 = torch.exp(-mx_1_2/sigma_z)
    mx_1 = mx_1_1 * mx_1_2

    mx_2_1 = torch.exp(-torch.abs(Y.repeat(1,n) - hat_Y.repeat(1,n).T)/sigma_y)
    mx_2 = mx_2_1 * mx_1_2

    mx_3 = mx_2.T

    mx_4_1 = torch.exp(-torch.abs(hat_Y.repeat(1,n) - hat_Y.repeat(1,n).T)/sigma_y)
    mx_4 = mx_4_1 * mx_1_2

    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss =  1/(n**2) * torch.sum(FF_mx)
    return loss

def find_loss_2(Y, hat_Y, Z, sigma_z, sigma_y):
    """
    Compute the MMD loss via Gaussian kernel.

    Inputs:
    - y_torch: PyTorch Tensor (batch_size) of training input. (X or Y)
    - gen_y_all_torch: PyTorch Tensor (batch_size, M) of generated data.
    - z_torch: PyTorch Tensor (batch_size, dimension_Z) of training input Z.
    - sigma_w: Float of the bandwith of the kernel.
    - sigma_u: Float of the bandwith of the kernel.
    - M: Number of training samples per batch.

    Outputs:
    - loss: PyTorch Tensor containing the MMD loss.
    """

    n = Z.shape[0]
    mx_1_1 = torch.exp(-(Y.repeat(1,n) - Y.repeat(1,n).T) ** 2/sigma_y)
    mx_1_2 = torch.linalg.vector_norm(Z.repeat(n,1,1) - torch.swapaxes(Z.repeat(n,1,1), 0, 1), ord = 2, dim = 2)
    # sigma = torch.median(mx_1_2)
    mx_1_2 = torch.exp(-(mx_1_2 ** 2)/sigma_z)
    mx_1 = mx_1_1 * mx_1_2

    mx_2_1 = torch.exp(-(Y.repeat(1,n) - hat_Y.repeat(1,n).T) ** 2/sigma_y)
    mx_2 = mx_2_1 * mx_1_2

    mx_3 = mx_2.T

    mx_4_1 = torch.exp(-(hat_Y.repeat(1,n) - hat_Y.repeat(1,n).T) ** 2/sigma_y)
    mx_4 = mx_4_1 * mx_1_2

    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss =  1/(n**2) * torch.sum(FF_mx)
    return loss

def train_ver3(
      G_zx, G_zy, # 修改：接收外部传入的模型
      X, Y, Z, X_test, Y_test, Z_test, M,
      noise_dimension, noise_type, G_lr, hidden_layer_size,
      DataLoader, BN_type, ReLU_coef,
      lambda_mean=0.5, mean_samples=20,
      epochs_num=50,  

      patience=5, min_delta=1e-5,

      sigma_z = 1, sigma_x = 1, sigma_y = 1,
      normal_ini = False,
      lambda_1 = 1, lambda_2 = 1, using_Gen = '1', wgt_decay = 0,
      lambda_3 = 0, drop_out_p = 0.2, noise_dimension_var = 1,
      lambda_4 = 0):
    """
    Train loop for GAN.

    Inputs:
    - X: PyTorch Tensor (sample_size, dimension_X) of training input.
    - Y: PyTorch Tensor (sample_size, dimension_Y) of training input.
    - Z: PyTorch Tensor (sample_size, dimension_Z) of training input.
    - X_test: PyTorch Tensor (sample_size, dimension_X) of test input.
    - Y_test: PyTorch Tensor (sample_size, dimension_Y) of test input.
    - Z_test: PyTorch Tensor (sample_size, dimension_Z) of test input.
    - noise_dimension: Integer giving the dimension of random noise Z.
    - noise_type: "normal", "unif" or "Cauchy", giving the reference distribution.
    - G_lr: Float giving the learning rate of the generator.
    - hidden_layer_size: Integer giving the size of the hidden layer of the generator.
    - DataLoader: DataLoader object used to generate training batches.
    - BN_type: 'True' or 'False' specifying whether batch normalization is included.
    - ReLU_coef: Float giving the coefficient of the Leaky ReLU layer.
    - epochs_num: Number of epochs over the training dataset to use for training.
    - sigma_z: Float of the bandwith of the kernel.
    - sigma_x: Float of the bandwith of the kernel.
    - sigma_y: Float of the bandwith of the kernel.
    - normal_ini: Boolean specifying whether to initialize the generator with normal initialization.
    - lambda_1: Float giving the coefficient of the MMD loss using Laplace kernel.
    - lambda_2: Float giving the coefficient of the MMD loss using Gaussian kernel. (not using)
    - using_Gen: '1' or '2' specifying whether to use the first or second generator.(not using)
    - wgt_decay: Float giving the weight decay. (L2 regularization)
    - lambda_3: Scalar giving the coefficient of the L1 regularization.
    - drop_out_p: Float giving the dropout probability.
    - M_train: Number of training samples per batch used in the Laplace or Gaussian kernel.

    Outputs:
    - G_zy: PyTorch Net giving the trained generator.
    - G_zx: PyTorch Net giving the trained generator.
    """

    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]


    if G_zy is None or G_zx is None:
        if using_Gen == '1':
            G_zy = Generator(input_dimension, output_dimension_y, noise_dimension, hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)
            G_zx = Generator(input_dimension, output_dimension_x, noise_dimension, hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)
        elif using_Gen == '2':
            G_zy = Generator_2(input_dimension, output_dimension_y, noise_dimension, hidden_layer_size, BN_type, ReLU_coef).to(device)
            G_zx = Generator_2(input_dimension, output_dimension_x, noise_dimension, hidden_layer_size, BN_type, ReLU_coef).to(device)
        
        # 仅在第一次初始化时进行正态分布初始化
        if normal_ini:
            for p in G_zy.parameters(): p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))
            for p in G_zx.parameters(): p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))

    #if using_Gen == '1':
        #G_zy = Generator(input_dimension, output_dimension_y, noise_dimension, hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)
        #G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
        #G_zx = Generator(input_dimension, output_dimension_x, noise_dimension, hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)
        #G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

    #elif using_Gen == '2':
        #G_zy = Generator_2(input_dimension, output_dimension_y, noise_dimension, hidden_layer_size, BN_type,ReLU_coef).to(device)
        #G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
        #G_zx = Generator_2(input_dimension, output_dimension_x, noise_dimension, hidden_layer_size, BN_type,ReLU_coef).to(device)
        #G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    
    # 无论是否是新模型，每一折都需要重新定义优化器来清空动能状态
    G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)


    iter_count = 0
    G_zy = G_zy.train()
    G_zx = G_zx.train()

    #if normal_ini:
        #for p in G_zy.parameters():
            #p.data = torch.randn(
                #p.shape, device=device,
                #dtype=torch.float32) / np.sqrt(float(hidden_layer_size * 2))

        #for p in G_zx.parameters():
            #p.data = torch.randn(
                #p.shape, device=device,
                #dtype=torch.float32) / np.sqrt(float(hidden_layer_size * 2))

    best_loss = float('inf')
    counter = 0

    for epoch in range(epochs_num):
        # print('EPOCH: ', (epoch+1))
        batch_count = 0
        G_zy = G_zy.train()
        G_zx = G_zx.train()
        
        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)
            Y_real = Y_real.to(device)
            Z_real = Z_real.to(device)
            Z_fake = Z_fake.to(device)

            # 1. 准备数据: (Batch_size) -> (Batch_size * mean_samples)
            batch_size = Z_real.shape[0]
        
            # 这里的 repeat_interleave 使得 Z 变成 [z1, z1, ..., z2, z2, ...]
            Z_repeated = Z_real.repeat_interleave(mean_samples, dim=0) 
        
            # 2. 生成大量噪声
            Noise_for_mean = sample_noise(Z_repeated.shape[0], noise_dimension, noise_type, input_var=noise_dimension_var).to(device)

            # Generate fake data
            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type, input_var = noise_dimension_var).to(device)
            Y_fake = G_zy(torch.cat((Z_real,Noise_fake),dim=1)).to(device)

            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type, input_var = noise_dimension_var).to(device)
            X_fake = G_zx(torch.cat((Z_real,Noise_fake),dim=1)).to(device)



            # 3. 通过生成器 (注意不需要计算梯度用于内部层，为了节省显存，但这里我们需要反向传播优化 generator，所以通常需要梯度)
            # 为了避免计算图过大，这里要小心。标准的做法是直接算。
        
            # 针对 G_zy (生成 Y) 的均值惩罚
            Y_generated_group = G_zy(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            # Reshape: (Batch, Samples, Dim)
            Y_generated_reshaped = Y_generated_group.reshape(batch_size, mean_samples, -1)
            # 计算均值: (Batch, Dim)
            Y_mean_pred = torch.mean(Y_generated_reshaped, dim=1)
        
            # 均值损失: 让预测的条件均值靠近真实的 Y_real
            loss_mean_y = torch.nn.functional.mse_loss(Y_mean_pred, Y_real)

            # 针对 G_zx (生成 X) 的均值惩罚 (逻辑同上)
            X_generated_group = G_zx(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            X_mean_pred = torch.mean(X_generated_group.reshape(batch_size, mean_samples, -1), dim=1)
            loss_mean_x = torch.nn.functional.mse_loss(X_mean_pred, X_real)

            # ==========================================
            # 原有的 Loss 计算与反向传播
            # ==========================================

            # Generator step
            g_zy_error = None
            G_zy_solver.zero_grad()

            g_zx_error = None
            G_zx_solver.zero_grad()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0

            for name, param in G_zy.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param, ord = 1)
                else :
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_y = lambda_1 * find_loss(Y_real, Y_fake, Z_real, sigma_z = sigma_z, sigma_y = sigma_y) + \
                         lambda_2 * find_loss_2(Y_real, Y_fake, Z_real, sigma_z = sigma_z, sigma_y = sigma_y) + \
                         lambda_3 * l1_regularization_rest_layers + \
                         lambda_4 * l1_regularization_first_layer
            # *** 加入新的 Loss ***
            g_zy_error = mmd_loss_y + lambda_mean * loss_mean_y

            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()

            #g_zy_error = None
            #G_zy_solver.zero_grad()

            #g_zx_error = None
            #G_zx_solver.zero_grad()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0

            for name, param in G_zx.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_x = lambda_1 * find_loss(X_real, X_fake, Z_real, sigma_z = sigma_z, sigma_y = sigma_x) + \
                         lambda_2 * find_loss_2(X_real, X_fake, Z_real, sigma_z = sigma_z, sigma_y = sigma_x) + \
                         lambda_3 * l1_regularization_rest_layers + \
                         lambda_4 * l1_regularization_first_layer

            g_zx_error = mmd_loss_x + lambda_mean * loss_mean_x
            
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()

            iter_count += 1
            batch_count += 1

        # 修改 2: 在 epoch 结束处记录当前总 loss (假设为 g_zx_error + g_zy_error)
        current_loss = g_zx_error + g_zy_error # 需要在循环里累加每个batch的loss
        # 早停逻辑
        if current_loss < best_loss - min_delta:
            best_loss = current_loss
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            # print(f"Early stopping at epoch {epoch}") # 调试时可开启
            break

    return G_zy, G_zx

def mGAN(n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=1000,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=500, k=2, boot_num=1000,
         noise_dimension = 10, hidden_layer_size = 512, normal_ini = False, preprocess = 'normalize',
         G_lr = 1e-5, using_orcale = False, lambda_1 = 1, lambda_2 = 1,
         lambda_mean=0.3, mean_samples=20,
         using_Gen = '1',
         boor_rv_type = "gaussian", wgt_decay = 0, lambda_3 = 1, drop_out_p = 0.2, noise_dimension_var = 1,
         noise_dimension_type = "normal", lambda_4 = 1):
    """
    Compute the test statistics

    Inputs:
    - Ax: Torch Tensor of shape (sample_size, dimension_X) giving the matrix to generate training data of X.
    - Ay: Torch Tensor of shape (sample_size, dimension_Y) giving the matrix to generate training data of Y.
    - n: Integer giving the number of samples to generate.
    - z_dim: Integer giving the dimension of Z.
    - simulation: 'type1error' or 'power'.
    - batch_size: Integer giving the batch size.
    - epochs_num: Number of epochs over the training dataset to use for training.
    - nstd: Float. standard deviation of the noise in the simulated data.
    - z_dist: 'gaussian' or 'uniform'.
    - x_dims: Integer giving the dimension of X.
    - y_dims: Integer giving the dimension of Y.
    - a_x: Float using in the alternative case. alpha_x.
    - M: Integer giving the number of training samples per batch used in the Laplace or Gaussian kernel.
    - k: Integer giving the number of cross-validation folds.
    - boot_num: Integer of the number of wild bootstrap when computing p-value.
    - noise_dimension: Integer giving the dimension of random noise Z.
    - hidden_layer_size: Integer giving the size of the hidden layer of the generator.
    - normal_ini: Boolean specifying whether to initialize the generator with normal initialization.
    - G_lr: Float giving the learning rate of the generator.
    - using_orcale: 'True' or 'False' specifying whether to use the orcale method.
    - lambda_1: Float giving the coefficient of the MMD loss using Laplace kernel.
    - lambda_2: Float giving the coefficient of the MMD loss using Gaussian kernel.
    - using_Gen: '1' or '2' specifying whether to use the first or second generator.
    - boor_rv_type: 'rademacher', 'gaussian'. type of the bootstrap random variable.
    - wgt_decay: Float giving the weight decay. (L2 regularization)
    - lambda_3: Scalar giving the coefficient of the L1 regularization.
    - drop_out_p: Float giving the dropout probability
    - exp_num: not using
    - M_train: Number of training samples per batch used in the Laplace or Gaussian kernel.

    Outputs:
    - p-value: Float of computed p-value.

    """
    if simulation == 'type1error':
        # generate samples x, y, z under null hypothesis - x and y are conditional independent
        sim_x, sim_y, sim_z = generate_samples_random(size=n, sType='H0', dx=x_dims, dy=y_dims, dz=z_dim, nstd=nstd, alpha_x=a_x,
                                  dist_z=z_dist, preprocess = preprocess)

    elif simulation == 'power':
        # generate samples x, y, z under alternative hypothesis - x and y are dependent
        sim_x, sim_y, sim_z = generate_samples_random(size=n, sType='H1', dx=x_dims, dy=y_dims, dz=z_dim, nstd=nstd,
                                  alpha_x=a_x, dist_z=z_dist, preprocess = preprocess)
    else:
        raise ValueError('Test does not exist.')

    x, y, z = sim_x.to(device), sim_y.to(device), sim_z.to(device)

    w_mx = torch.linalg.vector_norm(z.repeat(n,1,1) - torch.swapaxes(z.repeat(n,1,1), 0, 1), ord = 1, dim = 2)
    sigma_w = torch.median(w_mx).item()

    u_mx = torch.exp(-torch.abs(y.repeat(1, n) - y.repeat(1, n).T) )
    sigma_u = torch.median(u_mx).item()

    v_mx = torch.exp(-torch.abs(x.repeat(1, n) - x.repeat(1, n).T) )
    sigma_v = torch.median(v_mx).item()

    test_size = int(n/k)
    stat_all = torch.zeros(k, 1)
    boot_temp_all = torch.zeros(k, boot_num)
    cur_k = 0

    # =============================================================
    # 修改点 1: 在循环外初始化模型，这样它们在 K 折之间可以重用权重
    # =============================================================
    if not using_orcale:
        # 这里需要根据你的 Generator 类定义来初始化
        # 假设使用的是代码中定义的 Generator 类
        G_zy = Generator(z_dim, y_dims, noise_dimension, hidden_layer_size, False,0.1, drop_out_p).to(device)
        
        
        G_zx = Generator(z_dim, x_dims, noise_dimension, hidden_layer_size, False,0.1, drop_out_p).to(device)
       
    # =============================================================

    for k_fold in range(k):
        k_fold_start = int(n/k * k_fold)
        k_fold_end = int(n/k * (k_fold+1))
        X_test, Y_test, Z_test = x[k_fold_start:k_fold_end], y[k_fold_start:k_fold_end], z[k_fold_start:k_fold_end]
        X_train, Y_train, Z_train = torch.cat((x[0:k_fold_start], x[k_fold_end:])), torch.cat((y[0:k_fold_start], y[k_fold_end:])), torch.cat((z[0:k_fold_start], z[k_fold_end:]))

        if (k == 1):
            X_train, Y_train, Z_train = X_test, Y_test, Z_test

        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)
        DataLoader_xyz = torch.utils.data.DataLoader(train_xyz, batch_size=batch_size, shuffle=True)
        
        if not using_orcale:
            current_epochs = epochs_num if k_fold == 0 else max(10, epochs_num // 5)
            # 注意：你需要确保你的 train_ver3 现在接受 G_zx 和 G_zy 作为输入参数
            # 如果 train_ver3 内部原本是自己实例化的，请改成接收传入的对象
            G_zy, G_zx = train_ver3(
                    G_zx=G_zx, G_zy=G_zy,
                    X = X_train, Y = Y_train, Z = Z_train, M = M,
                    X_test = X_test, Y_test = Y_test, Z_test = Z_test,
                    noise_dimension = noise_dimension, noise_type = noise_dimension_type,
                    G_lr = G_lr, hidden_layer_size = hidden_layer_size,
                    DataLoader = DataLoader_xyz, BN_type = False, ReLU_coef = 0.1,

                    lambda_mean=lambda_mean, mean_samples=mean_samples,

                    epochs_num=current_epochs, sigma_z = sigma_w, sigma_x = sigma_v, sigma_y = sigma_u,
                    normal_ini = normal_ini, lambda_1 = lambda_1, lambda_2 = lambda_2,
                    using_Gen = using_Gen, wgt_decay = wgt_decay, lambda_3 = lambda_3,
                    drop_out_p = drop_out_p, noise_dimension_var = noise_dimension_var,
                    lambda_4 = lambda_4)


        dataset_test = DatasetSelect(X_test, Y_test, Z_test)
        dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=True)

        gen_x_all = torch.zeros(test_size, M, device=device)
        gen_y_all = torch.zeros(test_size, M, device=device)
        z_all     = torch.zeros(test_size, z_dim, device=device)
        x_all     = torch.zeros(test_size, x_dims, device=device)
        y_all     = torch.zeros(test_size, y_dims, device=device)


        cur_itr = 0
        G_zx = G_zx.eval()
        G_zy = G_zy.eval()
        for i, (x_test, y_test, z_test) in enumerate(dataloader_test):
            z_test_temp = z_test.repeat(M,1)
            if not using_orcale:
                z_test_temp = z_test_temp.to(device)
                Noise_fake = sample_noise(z_test_temp.size()[0], noise_dimension, "normal").to(device)
                with torch.no_grad():
                    fake_x = G_zx(torch.cat((z_test_temp, Noise_fake),dim=1)).reshape(1, -1)

                Noise_fake = sample_noise(z_test_temp.size()[0], noise_dimension, "normal").to(device)
                with torch.no_grad():
                    fake_y = G_zy(torch.cat((z_test_temp, Noise_fake),dim=1)).reshape(1, -1)
            elif using_orcale:
                if simulation == 'type1error':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(z_test_temp, size=M, sType='H0', dx=x_dims, dy=y_dims, dz=z_dim, nstd=nstd, alpha_x=a_x,
                                      dist_z=z_dist)
                elif simulation == 'power':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(z_test_temp, size=M, sType='H1', dx=x_dims, dy=y_dims, dz=z_dim, nstd=nstd, alpha_x=a_x,
                                      dist_z=z_dist)

            gen_x_all[cur_itr,:] = fake_x.detach().reshape(-1)
            gen_y_all[cur_itr,:] = fake_y.detach().reshape(-1)
            x_all[cur_itr,:] = x_test
            y_all[cur_itr,:] = y_test
            z_all[cur_itr,:] = z_test
            cur_itr = cur_itr + 1
        cur_stat, cur_boot_temp = get_p_value_stat_1(boot_num, M, test_size, gen_x_all.to(device), gen_y_all.to(device),
                                x_all.to(device), y_all.to(device), z_all.to(device), sigma_w, sigma_u, sigma_v,
                                boor_rv_type)
        
        stat_all[cur_k,:] = cur_stat
        boot_temp_all[cur_k,:] = torch.from_numpy(cur_boot_temp)
        cur_k = cur_k + 1

        if using_orcale:
            gen_x_mean = torch.mean(gen_x_all, dim=1).reshape(-1, 1)
            gen_y_mean = torch.mean(gen_y_all, dim=1).reshape(-1, 1)

            mse_x = torch.mean((gen_x_mean - x_all) ** 2).item()
            mse_y = torch.mean((gen_y_mean - y_all) ** 2).item()

            print(f'Test MSE x [{mse_x}], MSE y [{mse_y}]')

    return np.mean(torch.mean(boot_temp_all, dim = 0).numpy() > torch.mean(stat_all).item() )


from joblib import Parallel, delayed
import multiprocessing
import numpy as np
from datetime import datetime

def run_experiment(params):
    # 1. 提取参数
    test = params["test"]
    sample_size = params["sample_size"]
    batch_size = params["batch_size"]
    z_dim = params["z_dim"]
    dx = params["dx"]
    dy = params["dy"]
    n_test = params["n_test"]
    epochs_num = params["epochs_num"]
    eps_std = params["eps_std"]
    dist_z = params["dist_z"]
    alpha_x = params["alpha_x"]
    m_value = params["m_value"]
    k_value = params["k_value"]
    j_value = params["j_value"]
    noise_dimension = params["noise_dimension"]
    noise_dimension_type = params["noise_dimension_type"]
    noise_dimension_var = params["noise_dimension_var"]
    hidden_layer_size = params["hidden_layer_size"]
    normal_ini = params["normal_ini"]
    preprocess = params["preprocess"]
    G_lr = params["G_lr"]
    alpha = params["alpha"]
    alpha1 = params["alpha1"]
    set_seeds = params["set_seeds"]
    using_orcale = params["using_orcale"]
    lambda_1 = params["lambda_1"]
    lambda_2 = params["lambda_2"]
    using_Gen = params["using_Gen"]
    boor_rv_type = params["boor_rv_type"]
    wgt_decay = params["wgt_decay"]
    lambda_3 = params["lambda_3"]
    lambda_4 = params["lambda_4"]
    drop_out_p = params["drop_out_p"]
    lambda_mean = params["lambda_mean"]
    mean_samples = params["mean_samples"]

    
    # 设置种子
    np.random.seed(set_seeds)
    torch.manual_seed(set_seeds)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(set_seeds)

    # 2. 并行配置
    # 获取可用核心数 (留2个核心给系统，避免死机)
    num_cores = 20
    if num_cores < 1: num_cores = 1
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 开始并行实验...")
    print(f"模式: {test}, 样本量: {sample_size}, 交叉验证折数: {k_value}, 模型均值对齐超参数: {lambda_mean}, 实验次数: {n_test}, 并行核数: {num_cores}")
    if test =='power':
        print(f"备择假设H_1下的模型参数 alpha_x: {alpha_x}")

    num_gpus = torch.cuda.device_count() 
    # 3. 使用 joblib 进行并行计算
    # 这里我们不再使用 for 循环，而是直接并行生成 p_value 列表
    p_values = Parallel(n_jobs=num_cores)(
        delayed(mGAN)(
            n=sample_size, z_dim=z_dim, simulation=test, batch_size=batch_size,
            epochs_num=epochs_num, nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy, 
            a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
            noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size, 
            normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr, 
            using_orcale=using_orcale, lambda_1=lambda_1, lambda_2=lambda_2, 

            lambda_mean=lambda_mean, mean_samples=mean_samples,

            using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay, 
            lambda_3=lambda_3, drop_out_p=drop_out_p,
            noise_dimension_var=noise_dimension_var, 
            noise_dimension_type=noise_dimension_type, lambda_4=lambda_4
        ) for _ in range(n_test) 
    )

    p_values = np.array(p_values)

    # 4. 计算两个 Level 的拒绝率 (Emp Rej Rate)
    # 计算 alpha 水平
    fp = [pval < alpha for pval in p_values]
    final_result = np.mean(fp)
    
    # 计算 alpha1 水平
    fp1 = [pval < alpha1 for pval in p_values]
    final_result1 = np.mean(fp1)

    # 5. 打印最终报告
    print(f"\n" + "="*50)
    print(f"实验类型: {test.upper()}")
    print(f"Z Dimension: {z_dim}")
    print(f"Emp Rej Rate: {final_result:.4f} (at significance level {alpha})")
    print(f"Emp Rej Rate: {final_result1:.4f} (at significance level {alpha1})")
    print("="*50 + "\n")

    return p_values

In [24]:

        param["sample_size"] = 400
        param["k_value"] = 2

        # 固定阈值看 type1error
        param["test"] = "type1error"
        param["n_test"] = 100
        param["lambda_mean"] = 0.5
        param["mean_samples"] = 30
        p_val_list = run_experiment(param)

        # H0 校准 power 用的阈值
        alpha_emp_10 = np.quantile(p_val_list, 0.10)
        alpha_emp_05 = np.quantile(p_val_list, 0.05)

        # 跑 power
        param["test"] = "power"
        param["alpha"] = alpha_emp_10
        param["alpha1"] = alpha_emp_05

        for alpha_x in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
            param["alpha_x"] = alpha_x
            p_val_list_h1 = run_experiment(param)

[11:29:37] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 100, 并行核数: 20

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1100 (at significance level 0.1)
Emp Rej Rate: 0.0600 (at significance level 0.05)

[11:34:35] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 100, 并行核数: 20

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1200 (at significance level 0.1)
Emp Rej Rate: 0.0800 (at significance level 0.05)

[11:40:07] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 100, 并行核数: 20
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1300 (at significance level 0.06870000000000001)
Emp Rej Rate: 0.0900 (at significance level 0.04235)

[11:46:05] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 100, 并行核数: 20
备择假设H_1下的模型参数 alpha_x: 0.08

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3800 (at significance level 0.06870000000000001)
Emp Rej Rate: 0.3100 (at significance level 0.04235)

[11:51:50] 开始并行实验.

In [7]:
# ============================================================
# 参数配置
# ============================================================
param = {
    "test": "power",              # ['type1error', 'power']
    "sample_size": 200,           # [200, 400, 600, 800, 1000]
    "batch_size": 128,            # [32, 64, 128, 256]
    "z_dim": 1,                   # [5, 50, 250]
    "dx": 1,
    "dy": 1,
    "n_test": 100,                # [200, 2000] number of trails to run
    "epochs_num": 100,            # [1000, 1500] number of epoch
    "eps_std": 1.0,               # epsilon std
    "dist_z": 'gaussian',         # ['laplace', 'gaussian']
    "alpha_x": 0.20,              # only used under alternative [0.05, 0.10, 0.15, 0.20, 0.25]
    "m_value": 100,               # [100, 200]
    "k_value": 8,                 # [2, 4, 6, 8] fold number
    "j_value": 1000,              # [1000, 2000]
    "noise_dimension": 5,         # [5, 10, 20]
    "noise_dimension_type": "normal",   # ["normal", "unif", "Cauchy"]
    "noise_dimension_var": 1,     # [1, 4, 9, 16]
    "hidden_layer_size": 1024,    # [64, 128, 256, 512, 1024]
    "normal_ini": False,
    "preprocess": 'scale',        # ['normalize', 'scale', 'None']
    "G_lr": 7e-5,
    "alpha": 0.1,
    "alpha1": 0.05,
    "set_seeds": 0,
    "using_orcale": False,
    "lambda_1": 1,
    "lambda_2": 1,
    "using_Gen": '1',
    "boor_rv_type": 'gaussian',
    "wgt_decay": 1e-5,
    "lambda_3": 1e-4,
    "lambda_4": 2e-5,
    "drop_out_p": 0.2,
    "is_sparse": True,
    "sparse_ratio": 0.05,
    "lambda_mean": 0.3,
    "mean_samples": 20
}

import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools
from tqdm import tqdm

# ============================================================
# [修改] 删除全局 device 变量，改为在 mGAN() 内部按 gpu_id 动态创建。
# 原代码：
#   enable_cuda = True
#   device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')
# ============================================================


def generate_samples_random(size=1000, sType='H0', dx=1, dy=1, dz=1, nstd=1.0,
                             alpha_x=0.05, preprocess="None", dist_z='gaussian'):
    num = size
    error_generator_x = TD.MultivariateNormal(torch.zeros(dx), 1 * torch.eye(dx))
    error_generator_y = TD.MultivariateNormal(torch.zeros(dy), 1 * torch.eye(dy))
    if dist_z == 'gaussian':
        z_generator = TD.MultivariateNormal(torch.zeros(dz), 1 * torch.eye(dz))
        Z = z_generator.sample((num,))
    elif dist_z == 'laplace':
        z_generator = TD.MultivariateNormal(torch.zeros(dz), 1 * torch.eye(dz))
        Z = z_generator.sample((num,))
    m = TD.Bernoulli(torch.tensor([alpha_x]))
    Axy = torch.ones((dx, dy)) * alpha_x
    if sType == 'H0':
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        X = Z + nstd * error_x
        Y = Z + nstd * error_y
    else:
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        delta = m.sample((num,))
        X = Z + (1 - delta) * nstd * error_x + delta * nstd * error_y
        Y = Z + nstd * error_y
    return X, Y, Z


def generate_samples_from_fixed_Z_random(Z, size=1000, sType='H0', dx=1, dy=1, dz=1,
                                          nstd=1.0, alpha_x=0.05, normalize=True,
                                          seed=None, dist_z='gaussian'):
    num = size
    error_generator_x = TD.MultivariateNormal(torch.zeros(dx), 1 * torch.eye(dx))
    error_generator_y = TD.MultivariateNormal(torch.zeros(dy), 1 * torch.eye(dy))
    Axy = torch.ones((dx, dy)) * alpha_x
    m = TD.Bernoulli(torch.tensor([alpha_x]))
    if sType == 'H0':
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        X = Z + nstd * error_x
        Y = Z + nstd * error_y
    else:
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        delta = m.sample((num,))
        X = Z + (1 - delta) * nstd * error_x + delta * nstd * error_y
        Y = Z + nstd * error_y
    return X, Y


def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch,
                        x_torch, y_torch, z_torch, sigma_w,
                        sigma_u=1, sigma_v=1, boor_rv_type="gaussian",
                        device=None):  # [修改] 新增 device 参数
    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2)
    w_mx = torch.exp(-w_mx / sigma_w)

    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(torch.exp(-torch.abs(
        gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u), dim=2)
    u_mx_3 = u_mx_2.T
    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(torch.exp(-torch.abs(
        gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)

    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1, n) - x_torch.repeat(1, n).T) / sigma_v)
    v_mx_2 = torch.mean(torch.exp(-torch.abs(
        gen_x_all_torch.repeat(n, 1, 1) - x_torch.repeat(1, n).reshape(n, n, 1)) / sigma_v), dim=2)
    v_mx_3 = v_mx_2.T
    gen_x_all_torch_rep = gen_x_all_torch.repeat(n, 1, 1)
    temp2_mx = gen_x_all_torch_rep[:, :, 0].T
    sum2_mx = torch.mean(torch.exp(-torch.abs(
        gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)

    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(torch.exp(-torch.abs(
            gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)
        sum_mx = sum_mx + temp_add_mx
        temp2_mx = gen_x_all_torch_rep[:, :, i].T
        temp2_add_mx = torch.mean(torch.exp(-torch.abs(
            gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)
        sum2_mx = sum2_mx + temp2_add_mx

    u_mx_4 = 1 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4
    v_mx_4 = 1 / M * sum2_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + v_mx_4

    FF_mx = u_mx * v_mx * w_mx * (1 - torch.eye(n).to(device))  # [修改]
    stat = 1 / (n - 1) * torch.sum(FF_mx).item()

    boottemp = np.array([])
    if boor_rv_type == "rademacher":
        eboot = torch.sign(torch.randn(n, boot_num)).to(device)  # [修改]
    elif boor_rv_type == "gaussian":
        eboot = torch.randn(n, boot_num).to(device)              # [修改]
    for bb in range(boot_num):
        random_mx = torch.matmul(eboot[:, bb].reshape(-1, 1), eboot[:, bb].reshape(-1, 1).T)
        bootmatrix = FF_mx * random_mx
        stat_boot = 1 / (n - 1) * torch.sum(bootmatrix).item()
        boottemp = np.append(boottemp, stat_boot)
    return stat, boottemp


class DatasetSelect(Dataset):
    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]


class DatasetSelect_GAN(torch.utils.data.Dataset):
    def __init__(self, X, Y, Z, batch_size):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return (self.X_real[index], self.Y_real[index], self.Z_real[index],
                self.Z_real[(self.batch_size + index) % self.sample_size])


class DatasetSelect_GAN_ver2(torch.utils.data.Dataset):
    def __init__(self, Y, Z, batch_size):
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = Z.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.Y_real[index], self.Z_real[index]


##### Auxilliary functions #####

def sample_noise(sample_size, noise_dimension, noise_type, input_var=1,
                 device=None):  # [修改] 新增 device 参数
    if noise_type == "normal":
        noise_generator = TD.MultivariateNormal(
            torch.zeros(noise_dimension).to(device),          # [修改]
            input_var * torch.eye(noise_dimension).to(device) # [修改]
        )
        Z = noise_generator.sample((sample_size,))
    if noise_type == "unif":
        Z = torch.rand(sample_size, noise_dimension)
    if noise_type == "Cauchy":
        Z = TD.Cauchy(torch.tensor([0.0]), torch.tensor([1.0])).sample(
            (sample_size, noise_dimension)).squeeze(2)
    return Z


##### GAN architecture #####

class Generator(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef, drop_out_p, drop_input=False):
        super(Generator, self).__init__()
        self.BN_type = BN_type
        self.ReLU_coef = ReLU_coef
        self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN3 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
        self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)
        self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc3 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)
        self.sigmoid = torch.nn.Sigmoid()
        self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out2 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out3 = torch.nn.Dropout(p=drop_out_p)
        self.drop_input = drop_input

    def forward(self, x):
        if self.BN_type:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
            x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
            # x = self.drop_out3(self.leakyReLU1(self.BN3(self.fc3(x))))
            x = self.fc_last(x)
        else:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
            x = self.drop_out2(self.leakyReLU1(self.fc2(x)))
            # x = self.drop_out3(self.leakyReLU1(self.fc3(x)))
            x = self.fc_last(x)
        return x


class NonFullyConnected_1(torch.nn.Module):
    def __init__(self, size_in, size_out, m, bias=True):
        super(NonFullyConnected_1, self).__init__()
        # [修改] 不在此处 .to(device)，由外层调用者统一 .to(device)
        self.linear = torch.nn.Linear(m * size_in, m * size_out, bias=bias)
        self.mask = functools.reduce(
            torch.block_diag,
            [torch.ones(size_out, size_in) for i in range(m)])

    def forward(self, x):
        # [修改] mask 动态跟随模型所在设备
        self.linear.weight.data *= self.mask.to(self.linear.weight.device)
        return self.linear(x)


class Generator_2(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef,
                 hidden_layer_depth=1, ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)
        self.linear_layers_from_input = torch.nn.Linear(
            self.input_dimension, ntargets_k * self.hidden_layer_sizes[0])
        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0], self.hidden_layer_sizes[0], ntargets_k)
            for i in range(len(self.hidden_layer_sizes))
        ])
        self.linear8 = torch.nn.Linear(
            self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

    def forward(self, input):
        if self.BN_type:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(self.BN1(output))
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(self.BN1(output))
        else:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(output)
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(output)
        return self.linear8(output)


##### Training procedures #####

def find_loss(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-torch.abs(Y.repeat(1, n) - Y.repeat(1, n).T) / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    mx_1_2 = torch.exp(-mx_1_2 / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-torch.abs(Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-torch.abs(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss


def find_loss_2(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-(Y.repeat(1, n) - Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=2, dim=2)
    mx_1_2 = torch.exp(-(mx_1_2 ** 2) / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-(Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss


def train_ver3(
        G_zx, G_zy,
        X, Y, Z, X_test, Y_test, Z_test, M,
        noise_dimension, noise_type, G_lr, hidden_layer_size,
        DataLoader, BN_type, ReLU_coef,
        lambda_mean=0.5, mean_samples=20,
        epochs_num=50, patience=5, min_delta=1e-5,
        sigma_z=1, sigma_x=1, sigma_y=1, normal_ini=False,
        lambda_1=1, lambda_2=1, using_Gen='1', wgt_decay=0,
        lambda_3=0, drop_out_p=0.2, noise_dimension_var=1, lambda_4=0,
        device=None):  # [修改] 新增 device 参数
    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]

    if G_zy is None or G_zx is None:
        if using_Gen == '1':
            G_zy = Generator(input_dimension, output_dimension_y, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改]
            G_zx = Generator(input_dimension, output_dimension_x, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改]
        elif using_Gen == '2':
            G_zy = Generator_2(input_dimension, output_dimension_y, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改]
            G_zx = Generator_2(input_dimension, output_dimension_x, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改]
        if normal_ini:
            for p in G_zy.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改]
            for p in G_zx.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改]

    # 无论是否是新模型，每一折都重新定义优化器以清空动量状态
    G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

    iter_count = 0
    G_zy = G_zy.train()
    G_zx = G_zx.train()

    best_loss = float('inf')
    counter = 0

    for epoch in range(epochs_num):
        batch_count = 0
        G_zy = G_zy.train()
        G_zx = G_zx.train()

        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)   # [修改]
            Y_real = Y_real.to(device)   # [修改]
            Z_real = Z_real.to(device)   # [修改]
            Z_fake = Z_fake.to(device)   # [修改]

            batch_size = Z_real.shape[0]
            Z_repeated = Z_real.repeat_interleave(mean_samples, dim=0)

            Noise_for_mean = sample_noise(Z_repeated.shape[0], noise_dimension, noise_type,
                                          input_var=noise_dimension_var, device=device).to(device)  # [修改]

            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)       # [修改]
            Y_fake = G_zy(torch.cat((Z_real, Noise_fake), dim=1)).to(device)
            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)       # [修改]
            X_fake = G_zx(torch.cat((Z_real, Noise_fake), dim=1)).to(device)

            Y_generated_group = G_zy(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            Y_generated_reshaped = Y_generated_group.reshape(batch_size, mean_samples, -1)
            Y_mean_pred = torch.mean(Y_generated_reshaped, dim=1)
            loss_mean_y = torch.nn.functional.mse_loss(Y_mean_pred, Y_real)

            X_generated_group = G_zx(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            X_mean_pred = torch.mean(X_generated_group.reshape(batch_size, mean_samples, -1), dim=1)
            loss_mean_x = torch.nn.functional.mse_loss(X_mean_pred, X_real)

            g_zy_error = None
            G_zy_solver.zero_grad()
            g_zx_error = None
            G_zx_solver.zero_grad()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param in G_zy.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_y = (lambda_1 * find_loss(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_2 * find_loss_2(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zy_error = mmd_loss_y + lambda_mean * loss_mean_y
            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param in G_zx.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_x = (lambda_1 * find_loss(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_2 * find_loss_2(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zx_error = mmd_loss_x + lambda_mean * loss_mean_x
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()

            iter_count += 1
            batch_count += 1

        current_loss = g_zx_error + g_zy_error
        if current_loss < best_loss - min_delta:
            best_loss = current_loss
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            break

    return G_zy, G_zx


def mGAN(n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=1000,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=500, k=2, boot_num=1000,
         noise_dimension=10, hidden_layer_size=512, normal_ini=False, preprocess='normalize',
         G_lr=1e-5, using_orcale=False, lambda_1=1, lambda_2=1,
         lambda_mean=0.3, mean_samples=20, using_Gen='1',
         boor_rv_type="gaussian", wgt_decay=0, lambda_3=1, drop_out_p=0.2,
         noise_dimension_var=1, noise_dimension_type="normal", lambda_4=1,
         gpu_id=0):  # [修改] 新增 gpu_id 参数
    """
    gpu_id: 该 worker 使用的 GPU 编号（0/1/2/3）。
    由 run_experiment() 通过 job_index % num_gpus 传入，
    使不同 worker 分配到不同 GPU，而非全部挤在 GPU 0 上。
    """
    # [修改] 在函数内部根据 gpu_id 创建局部 device
    enable_cuda = True
    if torch.cuda.is_available() and enable_cuda:
        device = torch.device(f'cuda:{gpu_id}')
    else:
        device = torch.device('cpu')

    if simulation == 'type1error':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n, sType='H0', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess)
    elif simulation == 'power':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n, sType='H1', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess)
    else:
        raise ValueError('Test does not exist.')

    x, y, z = sim_x.to(device), sim_y.to(device), sim_z.to(device)

    w_mx = torch.linalg.vector_norm(
        z.repeat(n, 1, 1) - torch.swapaxes(z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    sigma_w = torch.median(w_mx).item()
    u_mx = torch.exp(-torch.abs(y.repeat(1, n) - y.repeat(1, n).T))
    sigma_u = torch.median(u_mx).item()
    v_mx = torch.exp(-torch.abs(x.repeat(1, n) - x.repeat(1, n).T))
    sigma_v = torch.median(v_mx).item()

    test_size = int(n / k)
    stat_all = torch.zeros(k, 1)
    boot_temp_all = torch.zeros(k, boot_num)
    cur_k = 0

    if not using_orcale:
        G_zy = Generator(z_dim, y_dims, noise_dimension, hidden_layer_size,
                         False, 0.1, drop_out_p).to(device)   # [修改]
        G_zx = Generator(z_dim, x_dims, noise_dimension, hidden_layer_size,
                         False, 0.1, drop_out_p).to(device)   # [修改]

    for k_fold in range(k):
        k_fold_start = int(n / k * k_fold)
        k_fold_end = int(n / k * (k_fold + 1))
        X_test = x[k_fold_start:k_fold_end]
        Y_test = y[k_fold_start:k_fold_end]
        Z_test = z[k_fold_start:k_fold_end]
        X_train = torch.cat((x[0:k_fold_start], x[k_fold_end:]))
        Y_train = torch.cat((y[0:k_fold_start], y[k_fold_end:]))
        Z_train = torch.cat((z[0:k_fold_start], z[k_fold_end:]))
        if k == 1:
            X_train, Y_train, Z_train = X_test, Y_test, Z_test

        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)
        DataLoader_xyz = torch.utils.data.DataLoader(train_xyz, batch_size=batch_size, shuffle=True)

        if not using_orcale:
            current_epochs = epochs_num if k_fold == 0 else max(10, epochs_num // 5)
            G_zy, G_zx = train_ver3(
                G_zx=G_zx, G_zy=G_zy,
                X=X_train, Y=Y_train, Z=Z_train, M=M,
                X_test=X_test, Y_test=Y_test, Z_test=Z_test,
                noise_dimension=noise_dimension, noise_type=noise_dimension_type,
                G_lr=G_lr, hidden_layer_size=hidden_layer_size,
                DataLoader=DataLoader_xyz, BN_type=False, ReLU_coef=0.1,
                lambda_mean=lambda_mean, mean_samples=mean_samples,
                epochs_num=epochs_num, sigma_z=sigma_w, sigma_x=sigma_v, sigma_y=sigma_u,
                normal_ini=normal_ini, lambda_1=lambda_1, lambda_2=lambda_2,
                using_Gen=using_Gen, wgt_decay=wgt_decay, lambda_3=lambda_3,
                drop_out_p=drop_out_p, noise_dimension_var=noise_dimension_var,
                lambda_4=lambda_4,
                device=device)   # [修改] 传入局部 device

        dataset_test = DatasetSelect(X_test, Y_test, Z_test)
        dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=True)

        gen_x_all = torch.zeros(test_size, M, device=device)   # [修改]
        gen_y_all = torch.zeros(test_size, M, device=device)   # [修改]
        z_all = torch.zeros(test_size, z_dim, device=device)   # [修改]
        x_all = torch.zeros(test_size, x_dims, device=device)  # [修改]
        y_all = torch.zeros(test_size, y_dims, device=device)  # [修改]
        cur_itr = 0
        G_zx = G_zx.eval()
        G_zy = G_zy.eval()

        for i, (x_test, y_test, z_test) in enumerate(dataloader_test):
            z_test_temp = z_test.repeat(M, 1)
            if not using_orcale:
                z_test_temp = z_test_temp.to(device)                                    # [修改]
                Noise_fake = sample_noise(z_test_temp.size()[0], noise_dimension,
                                          "normal", device=device).to(device)            # [修改]
                with torch.no_grad():
                    fake_x = G_zx(torch.cat((z_test_temp, Noise_fake), dim=1)).reshape(1, -1)
                Noise_fake = sample_noise(z_test_temp.size()[0], noise_dimension,
                                          "normal", device=device).to(device)            # [修改]
                with torch.no_grad():
                    fake_y = G_zy(torch.cat((z_test_temp, Noise_fake), dim=1)).reshape(1, -1)
            elif using_orcale:
                if simulation == 'type1error':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp, size=M, sType='H0', dx=x_dims, dy=y_dims,
                        dz=z_dim, nstd=nstd, alpha_x=a_x, dist_z=z_dist)
                elif simulation == 'power':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp, size=M, sType='H1', dx=x_dims, dy=y_dims,
                        dz=z_dim, nstd=nstd, alpha_x=a_x, dist_z=z_dist)

            gen_x_all[cur_itr, :] = fake_x.detach().reshape(-1)
            gen_y_all[cur_itr, :] = fake_y.detach().reshape(-1)
            x_all[cur_itr, :] = x_test
            y_all[cur_itr, :] = y_test
            z_all[cur_itr, :] = z_test
            cur_itr = cur_itr + 1

        cur_stat, cur_boot_temp = get_p_value_stat_1(
            boot_num, M, test_size,
            gen_x_all.to(device), gen_y_all.to(device),
            x_all.to(device), y_all.to(device), z_all.to(device),
            sigma_w, sigma_u, sigma_v, boor_rv_type,
            device=device)   # [修改] 传入局部 device

        stat_all[cur_k, :] = cur_stat
        boot_temp_all[cur_k, :] = torch.from_numpy(cur_boot_temp)
        cur_k = cur_k + 1

    if using_orcale:
        gen_x_mean = torch.mean(gen_x_all, dim=1).reshape(-1, 1)
        gen_y_mean = torch.mean(gen_y_all, dim=1).reshape(-1, 1)
        mse_x = torch.mean((gen_x_mean - x_all) ** 2).item()
        mse_y = torch.mean((gen_y_mean - y_all) ** 2).item()
        print(f'Test MSE x [{mse_x}], MSE y [{mse_y}]')

    return np.mean(torch.mean(boot_temp_all, dim=0).numpy() > torch.mean(stat_all).item())


# ============================================================
# 并行部分
# ============================================================
from joblib import Parallel, delayed
import multiprocessing
import numpy as np
from datetime import datetime


def run_experiment(params):
    # 1. 提取参数
    test                 = params["test"]
    sample_size          = params["sample_size"]
    batch_size           = params["batch_size"]
    z_dim                = params["z_dim"]
    dx                   = params["dx"]
    dy                   = params["dy"]
    n_test               = params["n_test"]
    epochs_num           = params["epochs_num"]
    eps_std              = params["eps_std"]
    dist_z               = params["dist_z"]
    alpha_x              = params["alpha_x"]
    m_value              = params["m_value"]
    k_value              = params["k_value"]
    j_value              = params["j_value"]
    noise_dimension      = params["noise_dimension"]
    noise_dimension_type = params["noise_dimension_type"]
    noise_dimension_var  = params["noise_dimension_var"]
    hidden_layer_size    = params["hidden_layer_size"]
    normal_ini           = params["normal_ini"]
    preprocess           = params["preprocess"]
    G_lr                 = params["G_lr"]
    alpha                = params["alpha"]
    alpha1               = params["alpha1"]
    set_seeds            = params["set_seeds"]
    using_orcale         = params["using_orcale"]
    lambda_1             = params["lambda_1"]
    lambda_2             = params["lambda_2"]
    using_Gen            = params["using_Gen"]
    boor_rv_type         = params["boor_rv_type"]
    wgt_decay            = params["wgt_decay"]
    lambda_3             = params["lambda_3"]
    lambda_4             = params["lambda_4"]
    drop_out_p           = params["drop_out_p"]
    lambda_mean          = params["lambda_mean"]
    mean_samples         = params["mean_samples"]

    np.random.seed(set_seeds)
    torch.manual_seed(set_seeds)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(set_seeds)

    # 2. [修改] 获取可用 GPU 数量，用于轮询分配
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    num_cores = min(28, n_test)
    if num_cores < 1:
        num_cores = 1

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 开始并行实验...")
    print(f"模式: {test}, 样本量: {sample_size}, 交叉验证折数: {k_value}, "
          f"模型均值对齐超参数: {lambda_mean}, 实验次数: {n_test}, "
          f"并行核数: {num_cores}, 可用GPU数: {num_gpus}")
    if test == 'power':
        print(f"备择假设H_1下的模型参数 alpha_x: {alpha_x}")

    # 3. [修改] 按 job_index % num_gpus 轮询分配 GPU
    #    job 0,4,8,...  → GPU 0
    #    job 1,5,9,...  → GPU 1
    #    job 2,6,10,... → GPU 2
    #    job 3,7,11,... → GPU 3
    p_values = Parallel(n_jobs=num_cores)(
        delayed(mGAN)(
            n=sample_size, z_dim=z_dim, simulation=test, batch_size=batch_size,
            epochs_num=epochs_num, nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
            a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
            noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
            normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
            using_orcale=using_orcale, lambda_1=lambda_1, lambda_2=lambda_2,
            lambda_mean=lambda_mean, mean_samples=mean_samples,
            using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
            lambda_3=lambda_3, drop_out_p=drop_out_p,
            noise_dimension_var=noise_dimension_var,
            noise_dimension_type=noise_dimension_type, lambda_4=lambda_4,
            gpu_id=job_index % num_gpus    # [修改] 轮询分配 GPU
        )
        for job_index, _ in enumerate(range(n_test))  # [修改] enumerate 获取 job_index
    )

    p_values = np.array(p_values)

    # 4. 计算拒绝率
    fp = [pval < alpha for pval in p_values]
    final_result = np.mean(fp)
    fp1 = [pval < alpha1 for pval in p_values]
    final_result1 = np.mean(fp1)

    # 5. 打印结果
    print(f"\n" + "=" * 50)
    print(f"实验类型: {test.upper()}")
    print(f"Z Dimension: {z_dim}")
    print(f"Emp Rej Rate: {final_result:.4f} (at significance level {alpha})")
    print(f"Emp Rej Rate: {final_result1:.4f} (at significance level {alpha1})")
    print("=" * 50 + "\n")

    return p_values

In [6]:
param["sample_size"] = 1000
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 1000
param["lambda_mean"] = 0.3
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.02, 0.04, 0.06]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[13:36:08] 开始并行实验...
模式: type1error, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 1000, 并行核数: 24, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.0950 (at significance level 0.1)
Emp Rej Rate: 0.0360 (at significance level 0.05)

[14:09:45] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 1000, 并行核数: 24, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.02

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1620 (at significance level 0.1039)
Emp Rej Rate: 0.1040 (at significance level 0.06795000000000001)

[14:43:04] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 1000, 并行核数: 24, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.04

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2980 (at significance level 0.1039)
Emp Rej Rate: 0.2280 (at significance level 0.06795000000000001)

[15:16:23] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 1000, 并行核数: 24, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.06

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5760 (at significance level 0.1039

In [2]:
param["sample_size"] = 1000
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 1000
param["lambda_mean"] = 0.3
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[14:32:38] 开始并行实验...
模式: type1error, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 1000, 并行核数: 16, 可用GPU数: 4


/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1160 (at significance level 0.1)
Emp Rej Rate: 0.0530 (at significance level 0.05)

[15:15:07] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 1000, 并行核数: 16, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.4100 (at significance level 0.09190000000000001)
Emp Rej Rate: 0.2870 (at significance level 0.0449)

[15:55:50] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 1000, 并行核数: 16, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.08

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.8190 (at significance level 0.09190000000000001)
Emp Rej Rate: 0.7160 (at significance level 0.0449)

[16:33:32] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 1000, 并行核数: 16, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.9460 (at significance level 0.09190000000000001)
Emp Rej Rate: 0.9110 (at significance level 0.0449)

[17:09:31] 开始并行实验...
模式: power, 样本量: 100

Process LokyProcess-1178:
Process LokyProcess-1177:
Process LokyProcess-1176:
Process LokyProcess-1170:
Process LokyProcess-1161:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py", line 528, in _process_worker
    with worker_exit_lock:
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/backend/synchronize.py", line 119, in __enter__
    return self._semlock.acquire()
KeyboardInterrupt
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*s

KeyboardInterrupt: 

In [8]:
param["sample_size"] = 1000
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 1000
param["lambda_mean"] = 0.5
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.02, 0.04, 0.06]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[15:57:06] 开始并行实验...
模式: type1error, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 1000, 并行核数: 28, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1310 (at significance level 0.1)
Emp Rej Rate: 0.0670 (at significance level 0.05)

[16:28:53] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 1000, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.02

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1470 (at significance level 0.077)
Emp Rej Rate: 0.0680 (at significance level 0.03080000000000001)

[17:00:35] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 1000, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.04

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3170 (at significance level 0.077)
Emp Rej Rate: 0.1840 (at significance level 0.03080000000000001)

[17:32:13] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 1000, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.06

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5840 (at significance level 0.077)
E

In [6]:
param["sample_size"] = 1000
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 1000
param["lambda_mean"] = 0.5
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[13:41:19] 开始并行实验...
模式: type1error, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 1000, 并行核数: 16, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1140 (at significance level 0.1)
Emp Rej Rate: 0.0590 (at significance level 0.05)

[14:28:01] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 1000, 并行核数: 16, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.4220 (at significance level 0.088)
Emp Rej Rate: 0.2920 (at significance level 0.04)

[15:14:15] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 1000, 并行核数: 16, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.08

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.8570 (at significance level 0.088)
Emp Rej Rate: 0.7530 (at significance level 0.04)

[15:59:34] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 1000, 并行核数: 16, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.9710 (at significance level 0.088)
Emp Rej Rate: 0.9350 (at signifi

Process LokyProcess-1658:
Process LokyProcess-1663:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py", line 528, in _process_worker
    with worker_exit_lock:
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/backend/synchronize.py", line 119, in __enter__
    return self._semlock.acquire()
KeyboardInterrupt
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/23099458d/venvs/myjupyter/lib/python3

KeyboardInterrupt: 

In [4]:
param["sample_size"] = 400
param["k_value"] = 2

        # 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 100
param["lambda_mean"] = 0
param["mean_samples"] = 30
p_val_list = run_experiment(param)

        # H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

        # 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[13:19:01] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 100, 并行核数: 20, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1700 (at significance level 0.1)
Emp Rej Rate: 0.0900 (at significance level 0.05)

[13:21:23] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2000 (at significance level 0.0594)
Emp Rej Rate: 0.1400 (at significance level 0.0308)

[13:23:40] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.08

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3100 (at significance level 0.0594)
Emp Rej Rate: 0.1700 (at significance level 0.0308)

[13:25:52] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.4100 (at significance level 0.0594)
Emp Rej Rate: 0.3600 (at significance lev

In [2]:
param["sample_size"] = 400
param["k_value"] = 2

        # 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 100
param["lambda_mean"] = 0.3
param["mean_samples"] = 30
p_val_list = run_experiment(param)

        # H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

        # 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[11:09:38] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 100, 并行核数: 20, 可用GPU数: 4


/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.0800 (at significance level 0.1)
Emp Rej Rate: 0.0500 (at significance level 0.05)

[11:11:38] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2600 (at significance level 0.13110000000000002)
Emp Rej Rate: 0.1100 (at significance level 0.0499)

[11:13:25] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.08

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5800 (at significance level 0.13110000000000002)
Emp Rej Rate: 0.3400 (at significance level 0.0499)

[11:15:20] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.6200 (at significance level 0.13110000000000002)
Emp Rej Rate: 0.4800 (at significance level 0.0499)

[11:17:25] 开始并行实验...
模式: power, 样本量: 400, 交叉验证

In [4]:
param["sample_size"] = 400
param["k_value"] = 2

        # 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 100
param["lambda_mean"] = 0.2
param["mean_samples"] = 30
p_val_list = run_experiment(param)

        # H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

        # 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[11:27:52] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.2, 实验次数: 100, 并行核数: 20, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.0900 (at significance level 0.1)
Emp Rej Rate: 0.0600 (at significance level 0.05)

[11:30:24] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.2, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2100 (at significance level 0.1056)
Emp Rej Rate: 0.0800 (at significance level 0.0439)

[11:33:19] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.2, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.08

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3600 (at significance level 0.1056)
Emp Rej Rate: 0.1500 (at significance level 0.0439)

[11:36:13] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.2, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.4900 (at significance level 0.1056)
Emp Rej Rate: 0.3800 (at signific